Install and Import Required Libraries
Run this in your first cell to install any necessary scraping and machine learning libraries.

In [2]:
# Run this cell to install required libraries if not already installed
%pip install pandas scikit-learn skl2onnx onnxruntime

You should consider upgrading via the 'c:\Users\DELL\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.



Download Directly in Your Notebook

Once the environment variables are injected, the official Python library automatically authenticates without requiring a kaggle.json file. Run this script directly inside your OpenShift AI notebook cell:

In [2]:
import os
import subprocess

# 1. Install the Kaggle CLI tool
subprocess.run(["pip", "install", "--quiet", "kaggle"])

# 2. Define the dataset slug (e.g., 50k IMDb reviews)
dataset_slug = "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
download_path = "./data"

# 3. Execute the secure download and unzip
os.makedirs(download_path, exist_ok=True)
subprocess.run([
    "kaggle", "datasets", "download", 
    "-d", dataset_slug, 
    "-p", download_path, 
    "--unzip"
])

print(f"Dataset downloaded successfully to: {download_path}")


Dataset downloaded successfully to: ./data


Load Kaggle Dataset

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load the 50K IMDB dataset from Kaggle
# Make sure 'IMDB Dataset.csv' is in the same directory as your notebook
df_kaggle = pd.read_csv('./data/IMDB Dataset.csv')

# Convert text labels ('positive'/'negative') to binary (1/0)
df_kaggle['label'] = df_kaggle['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

print("Dataset shape:", df_kaggle.shape)
df_kaggle.head()

Dataset shape: (50000, 3)


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


Train the Sentiment Analysis Model
We train the model using the 50K dataset with TF-IDF vectorization and Logistic Regression.

In [4]:
X = df_kaggle['review']
y = df_kaggle['label']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorize text using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate model performance
preds = model.predict(X_test_vec)
print(f"Model Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")

Model Accuracy: 89.10%


Save and Export Your Trained Model
Inside your workbench notebook, serialize your trained model and vectorizer into files using joblib or pickle:

In [5]:
import os
from sklearn.pipeline import Pipeline
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

# 1. Create the 'model' directory if it does not exist
os.makedirs('model', exist_ok=True)

# 2. Combine vectorizer and model into a single pipeline
pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('classifier', model)
])

# 3. Define the input signature (expecting raw text strings as input)
initial_type = [("text_input", StringTensorType([None, 1]))]

# 4. Convert the scikit-learn pipeline to ONNX format
onnx_model = convert_sklearn(pipeline, initial_types=initial_type, target_opset=12)

# 5. Save the ONNX model inside the 'model' directory
onnx_filename = os.path.join('model', 'movie_review_sentiment_model.onnx')
with open(onnx_filename, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"Model and vectorizer successfully saved as ONNX: {onnx_filename}!")

Model and vectorizer successfully saved as ONNX: model\movie_review_sentiment_model.onnx!


Predict Movie Category (Good or Bad) Using an IMDb Review Comments
This function combines the scraper and your trained model to analyze an IMDb Review Comments and output an overall verdict.

In [ ]:
import os
import numpy as np
import pandas as pd
import onnxruntime as ort

def classify_movie(reviews):
    print(f"Analyzing reviews for the movie...")
    
    if not reviews:
        return "No reviews found or access was blocked by IMDb."
    
    # Path to the saved ONNX model inside the 'model' directory
    model_path = os.path.join('model', 'movie_review_sentiment_model.onnx')
    
    if not os.path.exists(model_path):
        return f"Error: Model not found at {model_path}. Please train and save the model first."
    
    # Load the ONNX model inference session
    session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    
    input_name = session.get_inputs()[0].name
    
    # Format reviews into a 2D NumPy array of strings with shape [N, 1] as required by the ONNX pipeline
    input_data = np.array(reviews, dtype=object).reshape(-1, 1)
    
    # Run inference through ONNX Runtime
    outputs = session.run(None, {input_name: input_data})
    predictions = outputs[0]  # Contains predicted labels (0 or 1)
    
    # Flatten predictions to ensure standard 1D iteration
    predictions_flat = np.array(predictions).flatten()
    
    # Calculate percentage of positive (good) reviews
    positive_count = sum(predictions_flat == 1)
    total_reviews = len(predictions_flat)
    positive_ratio = (positive_count / total_reviews) * 100
    
    # Determine overall movie category
    verdict = "Good Movie 😊" if positive_ratio >= 50 else "Bad Movie 😞"
    
    print(f"\n--- Results ---")
    print(f"Total Reviews Analyzed: {total_reviews}")
    print(f"Positive Reviews: {positive_ratio:.1f}%")
    print(f"Overall Categorization: {verdict}\n")
    
    # Return sample breakdown DataFrame
    results_df = pd.DataFrame({
        "Review": reviews,
        "Prediction": ["Good" if p == 1 else "Bad" for p in predictions_flat]
    })
    return results_df

#--- Example Testing ---
sample_reviews = [
    "An absolute masterpiece with incredible storytelling and acting.",
    "Terrible waste of time, boring plot."
]
classify_movie(sample_reviews)

ImportError: DLL load failed while importing onnxruntime_pybind11_state: A dynamic link library (DLL) initialization routine failed.